# Notebook 02: Radar Diffusion Nowcaster — Evaluation

Visualise model predictions vs observations and compute skill scores.

**Prerequisites:**
- Stage 3 complete: `data/processed/radar.zarr` with ≥90 days of data
- Stage 4 complete: `checkpoints/nowcaster/latest.pt` exists
- `python scripts/evaluate.py` has been run → `results/evaluation_report.json`

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
import xarray as xr

from src.model.unet import ConditionedUNet
from src.model.diffusion import GaussianDiffusion
from src.data.radar_dataset import RadarDataset, load_stats, CONTEXT_FRAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Load evaluation report
report_path = ROOT / 'results/evaluation_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(json.dumps(report, indent=2))
else:
    print('Run: python scripts/evaluate.py first')

In [ ]:
# Load model
ckpt_path = ROOT / 'checkpoints/nowcaster/latest.pt'
state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
unet = ConditionedUNet(context_frames=CONTEXT_FRAMES, use_checkpoint=False)
diffusion = GaussianDiffusion(unet)
diffusion.load_state_dict(state['model'])
diffusion = diffusion.to(device).eval()
print(f'Model loaded from step {state["step"]}')

In [ ]:
# Visualise a heavy-rain event: pick a test sample with high observed rain
stats = load_stats()

def denorm(x):
    return np.expm1(np.clip(x * stats['log_std'] + stats['log_mean'], 0, None))

test_ds = RadarDataset('test', context_frames=CONTEXT_FRAMES)

# Find a heavy-rain sample
heavy_idx = None
for i in range(min(500, len(test_ds))):
    sample = test_ds[i]
    tgt_mm = denorm(sample['target'].numpy())
    if tgt_mm.max() >= 20.0:
        heavy_idx = i
        break

if heavy_idx is None:
    heavy_idx = 0
    print('No heavy-rain event found in first 500 samples; using idx=0')
else:
    print(f'Heavy-rain event at test index {heavy_idx}')

sample = test_ds[heavy_idx]
ctx = sample['context'].unsqueeze(0).to(device)
tgt_mm = denorm(sample['target'].numpy()[0])

with torch.no_grad():
    ens = diffusion.ensemble_sample(ctx, n_members=8, eta=1.0)
    ens_np = ens.squeeze(0).squeeze(2).cpu().numpy()

ens_mm = denorm(ens_np)
ens_mean = ens_mm.mean(axis=0)
ens_std  = ens_mm.std(axis=0)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
kw = dict(origin='upper', vmin=0, vmax=30, cmap=plt.cm.Blues)

im = axes[0].imshow(tgt_mm, **kw)
axes[0].set_title('Observed (ground truth)')
plt.colorbar(im, ax=axes[0], label='mm/hr')

im = axes[1].imshow(ens_mean, **kw)
axes[1].set_title('Ensemble mean')
plt.colorbar(im, ax=axes[1], label='mm/hr')

im = axes[2].imshow(ens_std, origin='upper', vmin=0, vmax=10, cmap=plt.cm.Oranges)
axes[2].set_title('Ensemble std (uncertainty)')
plt.colorbar(im, ax=axes[2], label='mm/hr')

persist_mm = denorm(sample['context'].numpy()[-1])
im = axes[3].imshow(persist_mm, **kw)
axes[3].set_title('Persistence (last frame)')
plt.colorbar(im, ax=axes[3], label='mm/hr')

plt.suptitle('Nowcaster vs Persistence — Heavy Rain Event', fontsize=14)
plt.tight_layout()
plt.savefig(ROOT / 'results/nowcast_heavy_rain_case.png', dpi=150)
plt.show()